# 07 — Final models on the test set

Each member's final model, fitted on train and scored once on the shared test set by AUC.

> **TODO (Oscar):** a line or two on what this notebook is for and the rule that nothing is tuned here.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

ROOT = Path.cwd().parent                  # run from report/
PROCESSED = ROOT / "data" / "processed"
TARGET = "default payment next month"

train = pd.read_csv(PROCESSED / "taiwan-train.csv", index_col="ID")
test  = pd.read_csv(PROCESSED / "taiwan-test.csv", index_col="ID")
y_train, y_test = train[TARGET], test[TARGET]

results = {}                              # model name -> test AUC

def test_auc(name, p_test):
    """Score one model's test-set probabilities of default."""
    results[name] = roc_auc_score(y_test, p_test)
    print(f"{name:<30} test AUC {results[name]:.4f}")

print(f"train {len(train):,}   test {len(test):,} customers, {y_test.mean():.2%} defaulted")

train 17,997   test 5,999 customers, 22.12% defaulted


## 1. Logistic regression (Oscar)

In [2]:
sys.path.append(str(ROOT / "Oscar"))
import prep                               # Oscar/prep.py: fixes 1-4 + ratio features

X_train = prep.prepare(train.drop(columns=TARGET), ratios=True)
X_test  = prep.prepare(test.drop(columns=TARGET), ratios=True)

# final model from 05: L2, C = 0.0316 (tuned on CV log loss)
logistic = make_pipeline(StandardScaler(),
                         LogisticRegression(C=0.0316, l1_ratio=0, solver="liblinear", max_iter=1000))
logistic.fit(X_train, y_train);

In [3]:
test_auc("Logistic regression", logistic.predict_proba(X_test)[:, 1])

Logistic regression            test AUC 0.7872


## 2. SVM (Oscar)

In [4]:
# final model from 06: linear SVM, squared hinge, C = 1e-4, Platt scaling for probabilities
# same features as the logistic regression above
svm = make_pipeline(StandardScaler(),
                    CalibratedClassifierCV(LinearSVC(C=1e-4, dual=True, max_iter=20000, random_state=2026),
                                           method="sigmoid", cv=5))
svm.fit(X_train, y_train);

In [5]:
test_auc("SVM", svm.predict_proba(X_test)[:, 1])

SVM                            test AUC 0.7903


## Results

In [6]:
pd.Series(results, name="test AUC").sort_values(ascending=False).round(4)

SVM                    0.7903
Logistic regression    0.7872
Name: test AUC, dtype: float64